# 08 — Check the analysis inputs

Run once for each resolution before notebook 09. This notebook checks the existing
panel and routing products. Model estimation and residual diagnostics belong in 09.
No routing is rebuilt. Save the notebook after running it.

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

CELL_SIZE = 200
LEGACY_100M = False
START_YEAR, END_YEAR, LAG_YEAR = 2016, 2025, 2015

PROJECT_DIR = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (p / "ANAL").is_dir())
if CELL_SIZE < 100 or CELL_SIZE % 100 or (LEGACY_100M and CELL_SIZE != 100):
    raise ValueError("Invalid resolution/profile.")
RESOLUTION = f"{CELL_SIZE}m"
DATA_DIR = PROJECT_DIR / "ANAL" / "data"
if not LEGACY_100M:
    DATA_DIR = DATA_DIR / RESOLUTION
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / f"raster_quarter_panel_{RESOLUTION}.parquet"
FIRMS_PATH = DATA_DIR / f"firms_assigned_{RESOLUTION}.geoparquet"
RASTER_PATH = DATA_DIR / f"raster_{RESOLUTION}_styria.geoparquet"
con = duckdb.connect()
sample_rows, diagnostic_rows, compatibility_mappings = [], [], []
CONFIG = {"legacy_100m": LEGACY_100M}
print(f"{RESOLUTION}; legacy={LEGACY_100M}; model period 2016–2025")

## 1. Required files and columns

In [ ]:
required = {
    PANEL_PATH: ["grid_id", "year", "quarter", "period", "births", "active_firms_tminus1", "municipality_id"],
    FIRMS_PATH: ["firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text"],
    RASTER_PATH: ["grid_id", "centroid_x", "centroid_y"],
}
features = {
    "accessibility_potentials": ["grid_id", "year", "quarter", "period", "own_cell_pop", "own_cell_firms",
                                "pop_access_15min", "existing_firms_access_15min"],
    "pedestrian_accessibility_quarter": ["grid_id", "year", "quarter", "own_cell_walk_pop",
        "own_cell_walk_firms", "walk_pop_10min", "walk_firms_10min", "walk_pt_routes_10min", "pt_ohne_haltestelle"],
    "nearest_infrastructure": ["grid_id", "year", "tt_motorway_exit_min"],
    "firm_accessibility_quarter": ["firm_id", "grid_id", "Fachgruppe_ID", "year", "quarter", "period",
        "included_in_lagged_stock", "own_cell_pop", "own_cell_firms", "own_cell_same_fachgruppe_firms",
        "pop_access_15min", "existing_firms_access_15min", "same_fachgruppe_firms_access_15min"],
}
for year in range(LAG_YEAR, END_YEAR + 1):
    for name, fields in features.items():
        if name == "firm_accessibility_quarter" and year < START_YEAR:
            continue
        required[FEATURE_DIR / str(year) / f"{name}_{RESOLUTION}.parquet"] = fields
for path, fields in required.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    available = set(pq.ParquetFile(path).schema_arrow.names)
    if LEGACY_100M and "grid_id_100m" in available:
        available.add("grid_id")
    missing = set(fields) - available
    if missing:
        raise ValueError(f"{path.name}: missing {sorted(missing)}")
print(f"All {len(required)} required input files have the needed columns.")

## 2. Panel coverage and counts

In [ ]:
panel_path = PANEL_PATH.as_posix().replace("'", "''")
con.execute(f"CREATE VIEW panel AS SELECT * FROM read_parquet('{panel_path}')")
coverage = con.execute("""
    SELECT year, count(*) AS rows, count(DISTINCT grid_id) AS cells,
           sum(births) AS births, avg(CAST(births=0 AS DOUBLE)) AS zero_share
    FROM panel WHERE year BETWEEN 2016 AND 2025 GROUP BY year ORDER BY year
""").df()
display(coverage)
print("Zeros are valid count observations. Missing routing values remain missing.")

## 3. Build the model tables once

Keep genuine zeros, exact previous-year population joins and lagged firm stocks.
Cox histories enter at max(founding, 2016Q1) and end at exit or 2025Q4.
Incomplete histories are excluded as a whole. Invalid routed dates and negative
neighbourhood masses stop preparation. No model is fitted here.

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(2**20), b""):
            digest.update(block)
    return digest.hexdigest()


def notebook_code_hash(path):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = "\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"]=="code")
    return hashlib.sha256(source.encode()).hexdigest()


def note(model, check, status, detail):
    diagnostic_rows.append(dict(model=model, check=check, status=status, detail=detail))


def columns(path):
    return set(pq.ParquetFile(path).schema_arrow.names)


def grid_column(path):
    names = columns(path)
    if "grid_id" in names: return "grid_id"
    if CONFIG["legacy_100m"] and "grid_id_100m" in names:
        item = {"file": str(path.relative_to(PROJECT_DIR)), "source": "grid_id_100m", "canonical": "grid_id"}
        if item not in compatibility_mappings: compatibility_mappings.append(item)
        return "grid_id_100m"
    raise ValueError(f"{path.name}: missing grid_id (legacy alias is allowed only in the legacy profile).")

def require_columns(path, required):
    available = columns(path)
    if "grid_id" in required:
        grid_column(path)
        available |= {"grid_id"}
    missing = sorted(set(required)-available)
    if missing: raise ValueError(f"{path}: missing columns {missing}")

def feature_view(name, filename, years):
    pieces = []
    for year in years:
        path = FEATURE_DIR / str(year) / filename
        key = grid_column(path)
        select = "*" if key == "grid_id" else "* EXCLUDE (grid_id_100m), grid_id_100m AS grid_id"
        sql_path = path.as_posix().replace("'", "''")
        pieces.append(f"SELECT {select} FROM read_parquet('{sql_path}')")
    con.execute(f"CREATE OR REPLACE TEMP VIEW {name} AS " + " UNION ALL BY NAME ".join(pieces))
    keys = "grid_id, year" if name == "nearest_features" else ("firm_id, year, quarter" if name == "firm_features" else "grid_id, year, quarter")
    duplicates = con.execute(f"SELECT count(*) FROM (SELECT {keys}, count(*) n FROM {name} GROUP BY {keys} HAVING n>1)").fetchone()[0]
    if duplicates: raise ValueError(f"{name}: {duplicates} duplicate source keys; joins would duplicate observations.")


def ring_difference(total, own, label):
    difference = total - own
    if difference.lt(-1e-8).any():
        raise ValueError(f"{label}: negative neighbourhood mass; inspect the source inputs.")
    return difference.clip(lower=0)

In [ ]:
FOUNDING_TERMS = ["log_own_firms", "log_own_pop", "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
                 "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10", "population_growth_yoy",
                 "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle"]
SURVIVAL_TERMS = ["log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15", "log_same_relative_ring_0_15",
                  "log_other_relative_ring_0_15", "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle", "calendar_year"]

def prepare_founding():
    feature_view("car_features", f"accessibility_potentials_{RESOLUTION}.parquet", range(LAG_YEAR, END_YEAR+1))
    feature_view("walk_features", f"pedestrian_accessibility_quarter_{RESOLUTION}.parquet", range(LAG_YEAR, END_YEAR+1))
    feature_view("nearest_features", f"nearest_infrastructure_{RESOLUTION}.parquet", range(LAG_YEAR, END_YEAR+1))
    panel_rows = con.execute(
        f"SELECT count(*) FROM read_parquet('{PANEL_PATH.as_posix()}') WHERE year BETWEEN {LAG_YEAR} AND {END_YEAR}"
    ).fetchone()[0]

    founding = con.execute(f"""
    SELECT p.grid_id, p.municipality_id, p.year, p.quarter, p.period,
           p.births, p.active_firms_tminus1,
           a.own_cell_pop, a.own_cell_firms,
           a.pop_access_15min, a.existing_firms_access_15min,
           w.own_cell_walk_pop, w.own_cell_walk_firms,
           w.walk_pop_10min, w.walk_firms_10min,
           w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
           n.tt_motorway_exit_min
    FROM read_parquet('{PANEL_PATH.as_posix()}') p
    LEFT JOIN car_features a
      USING (grid_id, year, quarter, period)
    LEFT JOIN walk_features w
      USING (grid_id, year, quarter, period)
    LEFT JOIN nearest_features n
      USING (grid_id, year)
    WHERE p.year BETWEEN {LAG_YEAR} AND {END_YEAR}
    ORDER BY p.grid_id, p.year, p.quarter
    """).df()

    if len(founding) != panel_rows:
        raise ValueError("The founding-data joins lost or duplicated panel rows.")
    if founding.duplicated(["grid_id", "year", "quarter"]).any():
        raise ValueError("Duplicate cell-quarter rows after the joins.")

    print(f"Panel input: {panel_rows:,} rows")
    print(f"Joined founding data: {founding.shape[0]:,} rows × {founding.shape[1]} columns")
    print(f"Cells: {founding['grid_id'].nunique():,}; quarters: {founding['period'].nunique()}")

    keys = founding[["grid_id", "year", "quarter", "period"]]
    if keys.isna().any().any() or not founding.quarter.isin([1, 2, 3, 4]).all():
        raise ValueError("Founding panel has missing/invalid time or cell keys.")
    if not founding.period.eq(founding.year.astype(str)+"Q"+founding.quarter.astype(str)).all():
        raise ValueError("period does not match year and quarter.")
    counts = founding.births.to_numpy(dtype=float)
    if not np.isfinite(counts).all() or (counts<0).any() or (counts != np.floor(counts)).any():
        raise ValueError("Births contain missing, negative or noninteger values.")
    raw = founding.loc[founding.year.ge(START_YEAR), ["grid_id", "year", "quarter", "municipality_id", "births"]]
    missing_quarters = len(raw.grid_id.unique())*4*(END_YEAR-START_YEAR+1)-len(raw)
    sample_rows.append({"model": "founding", "stage": "raw_panel", "rows": int(len(raw)), "units": int(raw.grid_id.nunique()),
                        "events": int(raw.births.sum()), "excluded": 0, "detail": f"missing cell-quarters={missing_quarters}"})
    del raw
    masses = ["active_firms_tminus1", "own_cell_pop", "own_cell_firms", "own_cell_walk_pop", "own_cell_walk_firms",
              "pop_access_15min", "existing_firms_access_15min", "walk_pop_10min", "walk_firms_10min", "walk_pt_routes_10min", "tt_motorway_exit_min"]
    if founding[masses].lt(0).any().any(): raise ValueError("Negative raw mass, stock or travel-time inputs.")
    if (~founding.pt_ohne_haltestelle.dropna().isin([0, 1])).any(): raise ValueError("Invalid PT indicator.")
    good = np.isfinite(founding[[*masses, "pt_ohne_haltestelle"]]).all(axis=1)
    dropped = int((~good & founding.year.ge(START_YEAR)).sum())
    founding = founding.loc[good].copy()
    # Non-overlapping neighbourhoods. The origin cell is removed.
    founding["car_pop_ring_0_15"] = ring_difference(founding["pop_access_15min"], founding["own_cell_pop"], "car_pop_ring_0_15")
    founding["car_firms_ring_0_15"] = ring_difference(founding["existing_firms_access_15min"], founding["own_cell_firms"], "car_firms_ring_0_15")
    founding["walk_pop_ring_0_10"] = ring_difference(founding["walk_pop_10min"], founding["own_cell_walk_pop"], "walk_pop_ring_0_10")
    founding["walk_firms_ring_0_10"] = ring_difference(founding["walk_firms_10min"], founding["own_cell_walk_firms"], "walk_firms_ring_0_10")

    founding["log_pop_access_ring_0_15"] = np.log1p(founding["car_pop_ring_0_15"])
    founding["log_firms_relative_car_ring_0_15"] = (
        np.log1p(founding["car_firms_ring_0_15"]) - founding["log_pop_access_ring_0_15"]
    )
    founding["log_walk_pop_ring_0_10"] = np.log1p(founding["walk_pop_ring_0_10"])
    founding["log_firms_relative_walk_ring_0_10"] = (
        np.log1p(founding["walk_firms_ring_0_10"]) - founding["log_walk_pop_ring_0_10"]
    )

    print("Created car and walking population-mass and relative-firm-density variables.")
    founding = founding.sort_values(["grid_id", "year", "quarter"])
    # Exact calendar join: missing intermediate quarters must not shift the lag to a different year.
    population_lookup = founding.set_index(["grid_id", "year", "quarter"])["own_cell_pop"]
    previous_keys = pd.MultiIndex.from_arrays([founding.grid_id, founding.year-1, founding.quarter], names=population_lookup.index.names)
    previous_population = pd.Series(population_lookup.reindex(previous_keys).to_numpy(), index=founding.index)

    founding["log_own_firms"] = np.log1p(founding["active_firms_tminus1"])
    founding["log_own_pop"] = np.log1p(previous_population)
    founding["population_growth_yoy"] = np.log1p(founding["own_cell_pop"]) - np.log1p(previous_population)
    founding["log_tt_motorway_exit"] = np.log1p(founding["tt_motorway_exit_min"])

    before = len(founding)
    founding = founding.loc[founding["year"] >= START_YEAR].copy()
    print(f"Removed {before-len(founding):,} lead-in rows from {LAG_YEAR}.")

    before = len(founding)
    founding = founding.dropna(subset=FOUNDING_TERMS).copy()
    excluded = dropped+before-len(founding)
    detail = f"routing={dropped}; missing lag/model values={before-len(founding)}"
    sample_rows.append({"model": "founding", "stage": "model_sample", "rows": int(len(founding)),
                        "units": int(founding.grid_id.nunique()), "events": int(founding.births.sum()),
                        "excluded": int(excluded), "zero_share": float(founding.births.eq(0).mean()) if len(founding) else None, "detail": detail})
    return founding[["grid_id", "municipality_id", "year", "quarter", "period", "births", *FOUNDING_TERMS]]

In [ ]:
def prepare_survival():
    feature_view("firm_features", f"firm_accessibility_quarter_{RESOLUTION}.parquet", range(START_YEAR, END_YEAR+1))
    feature_view("walk_features", f"pedestrian_accessibility_quarter_{RESOLUTION}.parquet", range(LAG_YEAR, END_YEAR+1))
    feature_view("nearest_features", f"nearest_infrastructure_{RESOLUTION}.parquet", range(LAG_YEAR, END_YEAR+1))
    raw_survival_rows = con.execute(
        f"SELECT count(*) FROM firm_features WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
    ).fetchone()[0]

    # The master file retains undated/out-of-window firms; routing excludes them.
    # Validate dates on the routed sample, before materializing the large interval join.
    routed_ids = con.execute(
        f"SELECT DISTINCT firm_id FROM firm_features WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
    ).df()["firm_id"]
    dates = pd.read_parquet(FIRMS_PATH, columns=["firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text"])
    dates = dates.rename(columns={"firm_id": "standort_id", "Sparte_ID": "sparte", "Sparte_Text": "sparte_name"})
    if dates.standort_id.isna().any() or dates.standort_id.duplicated().any(): raise ValueError("Firm identifiers are missing or duplicated.")
    in_sample = dates.standort_id.isin(routed_ids)
    coverage = {
        "source_firms": int(len(dates)), "routed_firms": int(len(routed_ids)),
        "unrouted_source_firms": int((~in_sample).sum()),
        "source_missing_founding_dates": int(dates.founding_date.isna().sum()),
        "source_invalid_exit_flags": int((~dates.exit_observed.isin([True, False, 0, 1])).sum()),
        "routed_missing_metadata": int((~routed_ids.isin(dates.standort_id)).sum()),
    }
    dates = dates.loc[in_sample].copy()
    for column in ["founding_date", "exit_date"]:
        dates[column] = pd.to_datetime(dates[column], errors="raise")
    missing_founding = int(dates.founding_date.isna().sum())
    invalid_flags = int((~dates.exit_observed.isin([True, False, 0, 1])).sum())
    missing_exit = int((dates.exit_observed.isin([True, 1]) & dates.exit_date.isna()).sum())
    reversed_dates = int((dates.exit_date < dates.founding_date).sum())
    coverage.update(routed_missing_founding_dates=missing_founding, routed_invalid_exit_flags=invalid_flags,
                    routed_observed_exits_without_dates=missing_exit, routed_exits_before_founding=reversed_dates)
    invalid = coverage["routed_missing_metadata"] + missing_founding + invalid_flags + missing_exit + reversed_dates
    note("survival", "source_date_coverage", "FAIL" if invalid else "INFO", json.dumps(coverage, sort_keys=True))
    if coverage["routed_missing_metadata"]:
        raise ValueError(f"Routed firms missing source metadata: firms={coverage['routed_missing_metadata']}.")
    if missing_founding or invalid_flags:
        raise ValueError(f"Invalid routed-firm metadata: missing_founding_dates={missing_founding}; invalid_exit_flags={invalid_flags}. "
                         "Repair the source metadata inside the air gap; no dates or censoring flags are imputed.")
    dates["exit_observed"] = dates.exit_observed.astype(bool)
    if missing_exit or reversed_dates:
        raise ValueError(f"Invalid routed-firm dates: observed_exits_without_dates={missing_exit}; exits_before_founding={reversed_dates}.")

    spells = con.execute(f"""
    SELECT a.firm_id AS standort_id, a.grid_id,
           CAST(a.Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID,
           a.year, a.quarter, a.period, a.included_in_lagged_stock,
           a.own_cell_pop, a.own_cell_firms, a.own_cell_same_fachgruppe_firms,
           a.pop_access_15min, a.existing_firms_access_15min,
           a.same_fachgruppe_firms_access_15min,
           w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
           n.tt_motorway_exit_min
    FROM firm_features a
    LEFT JOIN walk_features w
      ON a.grid_id=w.grid_id AND a.year=w.year AND a.quarter=w.quarter
    LEFT JOIN nearest_features n
      ON a.grid_id=n.grid_id AND a.year=n.year
    WHERE a.year BETWEEN {START_YEAR} AND {END_YEAR}
    ORDER BY a.firm_id, a.year, a.quarter
    """).df()

    if len(spells) != raw_survival_rows: raise ValueError("The survival joins lost or duplicated rows.")
    if spells.duplicated(["standort_id", "year", "quarter"]).any(): raise ValueError("Duplicate firm-quarter rows.")
    print(f"Firm-quarter input: {raw_survival_rows:,} rows")
    print(f"Joined survival data: {spells.shape[0]:,} rows × {spells.shape[1]} columns")
    print(f"Locations: {spells['standort_id'].nunique():,}")

    initial_rows, initial_units = len(spells), spells.standort_id.nunique()
    if spells[["standort_id", "grid_id", "year", "quarter", "period"]].isna().any().any():
        raise ValueError("Missing survival observation keys.")
    if not spells.quarter.isin([1,2,3,4]).all() or not spells.period.eq(spells.year.astype(str)+"Q"+spells.quarter.astype(str)).all():
        raise ValueError("Survival period does not match year/quarter.")
    spells = spells.merge(dates, on="standort_id", how="left", validate="many_to_one")
    if spells[["founding_date", "sparte", "sparte_name"]].isna().any().any(): raise ValueError("Routed firms lack dates/official sector labels.")
    current = pd.PeriodIndex(spells.period, freq="Q").asi8
    founding_quarter = spells.founding_date.dt.to_period("Q").array.asi8
    exit_quarter = spells.exit_date.dt.to_period("Q")
    end_period = pd.Period(f"{END_YEAR}Q4", freq="Q")
    effective_exit = spells.exit_observed & spells.exit_date.le(end_period.end_time)
    if (current < founding_quarter).any() or (effective_exit & spells.period.gt(exit_quarter.astype("string"))).any():
        raise ValueError("Survival products contain rows before founding or after observed exit.")
    spells["start"], spells["stop"] = current-founding_quarter, current-founding_quarter+1
    spells["event"] = (effective_exit & spells.period.eq(exit_quarter.astype("string"))).astype("int8")
    numeric = ["own_cell_pop", "own_cell_firms", "own_cell_same_fachgruppe_firms", "pop_access_15min",
               "existing_firms_access_15min", "same_fachgruppe_firms_access_15min", "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min", "included_in_lagged_stock"]
    if spells[numeric].lt(0).any().any(): raise ValueError("Negative survival source covariates.")
    for column in ["included_in_lagged_stock", "pt_ohne_haltestelle"]:
        if not spells[column].dropna().isin([0, 1]).all(): raise ValueError(f"Invalid {column} indicator.")
    bad = ~np.isfinite(spells[numeric]).all(axis=1) | spells.Fachgruppe_ID.isna()
    bad_ids = set(spells.loc[bad, "standort_id"])
    sequence = spells.year*4+spells.quarter
    gaps = sequence.groupby(spells.standort_id).diff().dropna().ne(1)
    bad_ids.update(spells.loc[gaps.index[gaps], "standort_id"])
    if spells.groupby("standort_id").event.sum().gt(1).any(): raise ValueError("More than one exit per firm.")
    # Missing first/last intervals must not silently become valid censoring.
    bounds = spells.groupby("standort_id", sort=False).agg(first=("period", "min"), last=("period", "max"), born=("founding_date", "first"), died=("exit_date", "first"), observed=("exit_observed", "first"))
    expected_first = bounds.born.dt.to_period("Q").astype("string").clip(lower=f"{START_YEAR}Q1")
    expected_last = bounds.died.dt.to_period("Q").astype("string").where(bounds.observed, f"{END_YEAR}Q4").fillna(f"{END_YEAR}Q4").clip(upper=f"{END_YEAR}Q4")
    bad_ids.update(bounds.index[bounds["first"].ne(expected_first) | bounds["last"].ne(expected_last)])
    spells = spells.loc[~spells.standort_id.isin(bad_ids)].copy()
    sample_rows.append({"model": "survival", "stage": "raw_intervals", "rows": int(initial_rows), "units": int(initial_units), "events": None, "excluded": 0, "detail": "before history exclusions"})
    focal = spells["included_in_lagged_stock"].astype("float64")
    spells["own_firms"] = ring_difference(spells["own_cell_firms"], focal, "survival_own_firms")
    spells["own_same"] = ring_difference(spells["own_cell_same_fachgruppe_firms"], focal, "survival_own_same")
    spells["own_other"] = ring_difference(spells["own_firms"], spells["own_same"], "survival_own_other")

    all_firms = ring_difference(spells["existing_firms_access_15min"], focal, "survival_all_firms")
    same_firms = ring_difference(spells["same_fachgruppe_firms_access_15min"], focal, "survival_same_firms")
    other_firms = ring_difference(all_firms, same_firms, "survival_other_firms")
    pop_ring = ring_difference(spells["pop_access_15min"], spells["own_cell_pop"], "survival_pop_ring")
    same_ring = ring_difference(same_firms, spells["own_same"], "survival_same_ring")
    other_ring = ring_difference(other_firms, spells["own_other"], "survival_other_ring")

    spells["log_own_pop"] = np.log1p(spells["own_cell_pop"])
    spells["log_own_same"] = np.log1p(spells["own_same"])
    spells["log_own_other"] = np.log1p(spells["own_other"])
    spells["log_pop_ring_0_15"] = np.log1p(pop_ring)
    spells["log_same_relative_ring_0_15"] = np.log1p(same_ring) - spells["log_pop_ring_0_15"]
    spells["log_other_relative_ring_0_15"] = np.log1p(other_ring) - spells["log_pop_ring_0_15"]
    spells["log_tt_motorway_exit"] = np.log1p(spells["tt_motorway_exit_min"])
    spells["calendar_year"] = spells["year"] - START_YEAR
    print("Removed the focal location and created the Cox-model covariates.")

    sample_rows.append({"model": "survival", "stage": "model_sample", "rows": int(len(spells)), "units": int(spells.standort_id.nunique()),
                        "events": int(spells.event.sum()), "excluded": int(initial_units-spells.standort_id.nunique()),
                        "detail": "excluded whole histories: routing, missing strata, gaps or incomplete boundaries"})
    return spells[["standort_id", "grid_id", "Fachgruppe_ID", "sparte", "sparte_name", "year", "period", "start", "stop", "event", *SURVIVAL_TERMS]]

In [ ]:
H3_BIRTHS_PATH = DATA_DIR / f"births_by_fachgruppe_{RESOLUTION}.parquet"
PREPARED_DIR = DATA_DIR / "cache" / "model_tables"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
founding = prepare_founding().reset_index(drop=True)
survival_frame = prepare_survival().reset_index(drop=True)
if founding.empty or survival_frame.empty:
    raise ValueError("An analysis sample is empty.")
sources = list(required)
if H3_BIRTHS_PATH.is_file():
    sources.append(H3_BIRTHS_PATH)
source_files = []
for path in sorted(sources):
    info = path.stat()
    source_files.append(dict(path=str(path.relative_to(PROJECT_DIR)), sha256=file_hash(path),
                             bytes=info.st_size, mtime_ns=info.st_mtime_ns))
tables = {}
for name, frame in [("founding",founding),("survival",survival_frame)]:
    path = PREPARED_DIR / f"{name}_frame.parquet"
    frame.to_parquet(path, index=False)
    tables[name] = dict(file=path.name, sha256=file_hash(path), rows=len(frame))
report = dict(status="completed", version="model-tables-1", resolution_m=CELL_SIZE,
              legacy_100m=LEGACY_100M, source_files=source_files, tables=tables,
              sample_description=sample_rows, diagnostics=diagnostic_rows,
              founding_terms=FOUNDING_TERMS, survival_terms=SURVIVAL_TERMS,
              preparation_notebook="08_200m_test_model_assumptions.ipynb",
              code_sha256=notebook_code_hash(PROJECT_DIR / "ANAL" / "08_200m_test_model_assumptions.ipynb"))
(PREPARED_DIR / "model_data_manifest.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame(sample_rows))
con.close()
print("Prepared model tables:", PREPARED_DIR)
print("Continue with 09. The cached rows stay local; only aggregate model results are exported.")